# General Relativity and Inference Methods
## GRAVITAS · self-contained notebook

Facundo Firmenich · [ORCID](https://orcid.org/0009-0002-6578-3811) · f.firmenich@cedesur.org

Run all extracts a verified payload, installs dependencies and reproduces calculations, tables and figures. Neither GitHub nor Drive is required; first installation needs Internet. Manuscripts and supplementary information are delivered separately and are not embedded in this notebook.

The graph offers a tour, play, pause, downloadable sources and claim boundaries. Cassini calculations are conditional on reconstructed inputs. No new physical bound or independent instrumental validation is claimed. iPhone/iPad access uses Colab in a browser, not a native executable.

Archived software v1.11: [10.5281/zenodo.24381485](https://doi.org/10.5281/zenodo.24381485) · Concept DOI: [10.5281/zenodo.24381484](https://doi.org/10.5281/zenodo.24381484).


DCAE: I = 0.90 corresponds to R = 10 and Δ = ln(10) by normalization; it is not a physical barrier or a probability. See docs/DCAE_NORMALIZATION_EN.md.


Canonical DCAE v0.6: ℛⱼ = 1/R; componentwise index, explicit protocol and triangulation without automatic identifiability. Run run.py dcae for the controls. See docs/DCAE_V06_EN.md.


FractalGo: independent software, [v0.1.0](https://github.com/FacundoFirmenich/fractal-go/tree/fe4fecdc44340fbadcbf1fdf3f226bad0a7661fa). DOI: 10.5281/zenodo.24383318.


In [ ]:
PAYLOAD_SHA256='284fe963669dcae302da7b98c4099a6dec70bd61b5bd122d52b1745ec59fcf82'
PAYLOAD='...'
from pathlib import Path
import base64,hashlib,zipfile,io,tempfile,subprocess,sys

def extract():
    raw=base64.b64decode(PAYLOAD)
    if hashlib.sha256(raw).hexdigest()!=PAYLOAD_SHA256:raise RuntimeError('Carga alterada / Payload integrity failure')
    work=Path(tempfile.mkdtemp(prefix='GRAVITAS_'))
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for item in archive.infolist():
            if not (work/item.filename).resolve().is_relative_to(work):raise RuntimeError('Ruta insegura / Unsafe archive path')
        archive.extractall(work)
    return work

WORK=extract()
from importlib.metadata import version,PackageNotFoundError
needed=[]
for line in (WORK/'requirements.txt').read_text().splitlines()+['ipywidgets==8.1.7']:
    name,wanted=line.split('==')
    try:
        if version(name)!=wanted:needed.append(line)
    except PackageNotFoundError:needed.append(line)
if needed:subprocess.check_call([sys.executable,'-m','pip','install',*needed])
print('Workspace / Carpeta:',WORK)
print('SHA-256:',PAYLOAD_SHA256)


## 1 · Run calculations, tables and figures
Includes information controls, Cassini, Gaia, QUMOND, DCAE and graph generation. Scientific predictive failures are retained even when the program exits successfully.

In [ ]:
subprocess.run([sys.executable,str(WORK/'run.py'),'all'],cwd=WORK,check=True)

## 2 · Explore the investigation

In [ ]:
from IPython.display import HTML, display
import html
graph=(WORK/'ui/epistemic/epistemic_graph_en.html').read_text()
display(HTML('<iframe style="width:100%;height:1100px;border:0" srcdoc="'+html.escape(graph,quote=True)+'"></iframe>'))

## 3 · Reproduce by block
Choose a mode and press the button. The selector does not execute arbitrary code.

In [ ]:
import ipywidgets as widgets
selector=widgets.Dropdown(options=[('FractalGo: topology', 'topology'), ('Fermi-LAT', 'fermi'), ('Posterior stability', 'stability'), ('Figures only', 'figures'), ('Calculations only', 'calculations'), ('Tables', 'tables'), ('Information', 'information'), ('DCAE v0.6 / COSMOS', 'dcae'), ('Graph and TikZ', 'graph')])
button=widgets.Button(description='Run')
output=widgets.Output()
def execute_block(_):
    button.disabled=True
    try:
        with output:
            output.clear_output()
            result=subprocess.run([sys.executable,str(WORK/'run.py'),selector.value],cwd=WORK,capture_output=True,text=True)
            print(result.stdout)
            print(result.stderr)
            result.check_returncode()
    finally:button.disabled=False
button.on_click(execute_block)
display(widgets.VBox([selector,button,output]))

## 4 · Save and download
The ZIP retains results, sources and records. Enable DOWNLOAD in Colab to download it.

In [ ]:
import shutil
archive=shutil.make_archive(str(WORK.parent/(WORK.name+'_outputs')),'zip',WORK)
print(archive)
DOWNLOAD=False
if DOWNLOAD:
    from google.colab import files
    files.download(archive)